# Task 3 - RAG with Unsloth Dynamic 4-bit Quantization
**Goal:** Build an end-to-end Retrieval-Augmented Generation (RAG) pipeline using an Unsloth dynamic 4-bit model in Google Colab.

In [ ]:
# Runtime pre-check (required for Unsloth 4-bit models)

import sys

import torch



print("Python:", sys.version.split()[0])

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():

    print("GPU:", torch.cuda.get_device_name(0))

else:

    raise RuntimeError(

        "This notebook requires a GPU runtime (Colab: Runtime > Change runtime type > T4/A100 GPU)."

    )


In [1]:
# Install required libraries
!pip -q install unsloth transformers accelerate bitsandbytes sentence-transformers faiss-cpu datasets

In [2]:
import os
import re
import torch
import numpy as np
from typing import List, Tuple
from unsloth import FastLanguageModel
from sentence_transformers import SentenceTransformer
import faiss

torch.backends.cuda.matmul.allow_tf32 = True
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

NotImplementedError: Unsloth currently only works on NVIDIA, AMD and Intel GPUs.

In [ ]:
# Load a dynamic 4-bit Unsloth model (change model if needed)
model_name = "unsloth/Llama-3.2-3B-Instruct-unsloth-bnb-4bit"
max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

print("Loaded model:", model_name)

In [ ]:
# Domain-specific source documents (replace with your own files/content)
documents = [
    "Generative AI can create text, images, audio, and code by learning patterns from large datasets.",
    "Retrieval-Augmented Generation combines information retrieval with language generation for grounded answers.",
    "Quantization reduces model memory usage by storing model weights in lower precision like 4-bit.",
    "Unsloth dynamic 4-bit quantization keeps critical parameters at higher precision to preserve quality while optimizing VRAM.",
    "RAG systems generally include document chunking, embedding generation, vector indexing, retrieval, and final response generation."
]

def chunk_text(text: str, chunk_size: int = 220, overlap: int = 40) -> List[str]:
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end]).strip()
        if chunk:
            chunks.append(chunk)
        start += max(1, chunk_size - overlap)
    return chunks

all_chunks = []
for d in documents:
    all_chunks.extend(chunk_text(d))

print("Total chunks:", len(all_chunks))
for i, c in enumerate(all_chunks[:3]):
    print(f"Chunk {i+1}:", c[:120], "...")

In [ ]:
# Build embeddings + FAISS index
embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=DEVICE)
embeddings = embed_model.encode(all_chunks, convert_to_numpy=True, batch_size=32, normalize_embeddings=True)

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings.astype(np.float32))

print("Embedding shape:", embeddings.shape)
print("FAISS vectors indexed:", index.ntotal)

In [ ]:
def retrieve(query: str, k: int = 3) -> List[Tuple[int, float, str]]:
    q_emb = embed_model.encode([query], convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)
    scores, ids = index.search(q_emb, k)
    out = []
    for rank, (idx, score) in enumerate(zip(ids[0], scores[0]), start=1):
        out.append((int(idx), float(score), all_chunks[int(idx)]))
    return out

def build_prompt(query: str, retrieved: List[Tuple[int, float, str]]) -> str:
    context = "\n".join([f"[{i+1}] {txt}" for i, (_, _, txt) in enumerate(retrieved)])
    return f"""You are a helpful AI assistant. Use only the context below to answer.
If context is insufficient, say so briefly.

Context:
{context}

Question: {query}
Answer:"""

In [ ]:
@torch.inference_mode()
def generate_answer(query: str, k: int = 3, max_new_tokens: int = 200):
    retrieved = retrieve(query, k=k)
    prompt = build_prompt(query, retrieved)

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_seq_length).to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.3,
        top_p=0.9,
        use_cache=True,
    )

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = text[len(prompt):].strip() if text.startswith(prompt) else text
    return answer, retrieved

query = "How does dynamic 4-bit quantization help in RAG systems?"
answer, retrieved = generate_answer(query, k=3)

print("Query:", query)
print("\nRetrieved Chunks:")
for i, (idx, score, txt) in enumerate(retrieved, start=1):
    print(f"{i}. idx={idx} score={score:.4f} -> {txt}")

print("\nGenerated Answer:")
print(answer)

## Notes for submission
- Replace toy `documents` with your own domain files (PDF/text/website exports).
- Add screenshots of Colab output: model loading, retrieval output, and final grounded answer.
- Mention GPU type and memory usage to show dynamic 4-bit efficiency.